In [9]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [10]:
# 기존 폴더 삭제 및 초기화
!rm -rf /content/alpha-beta-CROWN
%cd /content

# 저장소 및 하위 모듈 클론
!git clone --recursive https://github.com/Verified-Intelligence/alpha-beta-CROWN.git

# Conda 환경 세팅 (에러 없이 필요 패키지 일괄 설치)
%cd /content/alpha-beta-CROWN
!conda env update -n base -f complete_verifier/environment.yaml

# 서브모듈(auto_LiRPA) 설치
%cd /content/alpha-beta-CROWN/auto_LiRPA
!pip install -e .

/content
Cloning into 'alpha-beta-CROWN'...
remote: Enumerating objects: 1854, done.
remote: Counting objects: 100% (642/642), done.
remote: Compressing objects: 100% (367/367), done.
remote: Total 1854 (delta 326), reused 389 (delta 272), pack-reused 1212 (from 2)
Receiving objects: 100% (1854/1854), 82.98 MiB | 17.81 MiB/s, done.
Resolving deltas: 100% (985/985), done.
Submodule 'auto_LiRPA' (https://github.com/Verified-Intelligence/auto_LiRPA.git) registered for path 'auto_LiRPA'
Cloning into '/content/alpha-beta-CROWN/auto_LiRPA'...
remote: Enumerating objects: 1296, done.        
remote: Counting objects: 100% (825/825), done.        
remote: Compressing objects: 100% (639/639), done.        
remote: Total 1296 (delta 454), reused 193 (delta 186), pack-reused 471 (from 3)        
Receiving objects: 100% (1296/1296), 39.48 MiB | 17.32 MiB/s, done.
Resolving deltas: 100% (614/614), done.
Submodule path 'auto_LiRPA': checked out 'a050a3d61f4cb68b108fa7f07dcb3e4d7ef304df'
/content/alp

In [11]:
import torch
import torch.nn as nn
import sys

# 1. 모델 클래스 정의를 외부 파일(model_defs.py)로 저장
model_def_code = """
import torch
import torch.nn as nn

class AssignmentModel(nn.Module):
    def __init__(self):
        super(AssignmentModel, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x
"""
with open("/content/alpha-beta-CROWN/complete_verifier/model_defs.py", "w") as f:
    f.write(model_def_code)

# 2. 모델 객체 생성 및 가중치(.pth) 저장
sys.path.append('/content/alpha-beta-CROWN/complete_verifier')
from model_defs import AssignmentModel

model = AssignmentModel()
torch.save(model.state_dict(), "/content/alpha-beta-CROWN/complete_verifier/external_model.pth")
print("✅ model_defs.py 및 external_model.pth 파일 생성 완료!")

✅ model_defs.py 및 external_model.pth 파일 생성 완료!


In [12]:
yaml_content = """
model:
  # 핵심 수정: model_defs.py 안의 AssignmentModel을 불러오도록 Customized 문법 사용
  name: Customized("model_defs", "AssignmentModel")
  path: external_model.pth
data:
  dataset: MNIST
  std: [1.0]
  mean: [0.0]
specification:
  epsilon: 0.01
solver:
  batch_size: 64
  beta-crown:
    iteration: 10
"""

with open("/content/alpha-beta-CROWN/complete_verifier/my_config.yaml", "w") as f:
    f.write(yaml_content)
print("✅ my_config.yaml 설정 파일 생성 완료!")

✅ my_config.yaml 설정 파일 생성 완료!


In [13]:
test_script = """
import subprocess
import sys

def main():
    print("[과제 #4] alpha-beta-CROWN 검증을 시작합니다... (시간이 조금 걸릴 수 있습니다)")
    cmd = [sys.executable, "abcrown.py", "--config", "my_config.yaml"]

    # 텍스트 출력을 그대로 가져옵니다.
    result = subprocess.run(cmd, capture_output=True, text=True)

    print("\\n===========================================")
    print("           [검증 표준 출력 (결과)]           ")
    print("===========================================")
    print(result.stdout)

    if result.stderr:
        print("\\n===========================================")
        print("               [검증 에러 로그]              ")
        print("===========================================")
        print(result.stderr)

if __name__ == "__main__":
    main()
"""

with open("/content/alpha-beta-CROWN/complete_verifier/test.py", "w") as f:
    f.write(test_script)

# 실행 폴더로 이동 후 test.py 구동
%cd /content/alpha-beta-CROWN/complete_verifier
!python test.py

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
100%|██████████| 100/100 [00:00<00:00, 190.63it/s]

  1%|          | 1/100 [00:00<00:00, 105.49it/s]

100%|██████████| 100/100 [00:00<00:00, 221.03it/s]

  1%|          | 1/100 [00:00<00:01, 94.59it/s]

  1%|          | 1/100 [00:00<00:00, 105.42it/s]

  1%|          | 1/100 [00:00<00:01, 84.17it/s]

100%|██████████| 100/100 [00:00<00:00, 198.63it/s]

100%|██████████| 100/100 [00:00<00:00, 241.58it/s]

  1%|          | 1/100 [00:00<00:00, 120.85it/s]

100%|██████████| 100/100 [00:00<00:00, 250.07it/s]

  1%|          | 1/100 [00:00<00:00, 124.02it/s]

100%|██████████| 100/100 [00:00<00:00, 251.60it/s]

  1%|          | 1/100 [00:00<00:00, 112.47it/s]

  1%|          | 1/100 [00:00<00:00, 124.99it/s]

  1%|          | 1/100 [00:00<00:00, 101.44it/s]

  1%|          | 1/100 [00:00<00:00, 134.47it/s]

100%|██████████| 100/100 [00:00<00:00, 255.38it/s]

100%|██████████| 100/100 [00:00<00:00, 250.47it/s]

100%|██████████| 100/100 [00:00<00:00, 234.21it/s]